In [ ]:
import os
import sys
import time
import sqlite3
import pandas as pd
from datetime import datetime

# パス設定
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config import PROJECT_DIR

start_time = time.time()

# DBパス設定
user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == 'nt' else os.path.expanduser("~"),
    "myenv310", PROJECT_DIR
)
db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# 必要カラム
columns = [
    "実行日", "台番号", 
    "RB駆け抜け判定", 
    "RBスルー数", 
    "RB駆け抜け後スルー数", 
    "RB駆け抜け以外スルー数"
]

# データ取得
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query(f'''
        SELECT {", ".join([f"[{col}]" for col in columns])}
        FROM result_table
        ORDER BY ROWID DESC
    ''', conn)

# 実行日で最新のみ抽出
df["実行日"] = pd.to_datetime(df["実行日"], errors="coerce")
df = df.dropna(subset=["実行日"])
max_date = df["実行日"].dt.date.max()
df_today = df[df["実行日"].dt.date == max_date].copy()
print(f"[INFO] 最新実行日: {max_date}, 件数: {len(df_today)}")

# ---------- 分岐処理 ----------
def split_sure_count(row):
    sure = row.get("RBスルー数")
    escape = row.get("RB駆け抜け判定")

    if pd.isna(sure):
        return None, None  # スルー数がなければ何もしない

    if escape == 1:
        return sure, None
    else:
        return None, sure

df_today[["RB駆け抜け後スルー数", "RB駆け抜け以外スルー数"]] = df_today.apply(
    split_sure_count, axis=1, result_type="expand"
)

# 結果ログ
n_escape = df_today["RB駆け抜け後スルー数"].notna().sum()
n_other = df_today["RB駆け抜け以外スルー数"].notna().sum()
print(f"[INFO] 駆け抜け後スルー数 記載: {n_escape} 件")
print(f"[INFO] 駆け抜け以外スルー数 記載: {n_other} 件")

# ---------- DB更新 ----------
with sqlite3.connect(db_path) as conn:
    cur = conn.cursor()

    update_sql = '''
        UPDATE result_table
        SET [RB駆け抜け後スルー数] = ?, [RB駆け抜け以外スルー数] = ?
        WHERE [台番号] = ? AND date([実行日]) = ?
    '''

    updated = 0
    for _, row in df_today.iterrows():
        cur.execute(update_sql, (
            row.get("RB駆け抜け後スルー数"),
            row.get("RB駆け抜け以外スルー数"),
            row["台番号"],
            row["実行日"].strftime('%Y-%m-%d')
        ))
        updated += 1

    conn.commit()

print(f"✅ スルー数の振り分け更新完了: {updated}件")
print(f"[INFO] 所要時間: {time.time() - start_time:.1f}秒")
